# Table 6.2 — levels of the categorical predictors (Chapter 6)

Genera la tabla de niveles de las cuatro variables estáticas del Capítulo 6 a partir de
`slope_units_final_mm.gpkg`, y escribe un archivo `.tex` listo para `\input{}` en Overleaf.

La idea es que **no escribas la tabla a mano**. Si algún día cambias el GeoPackage o el
criterio de exclusión, vuelves a correr el notebook y la tabla se regenera sola, en vez de
quedarse desincronizada del texto sin que nadie se dé cuenta.

Lo que produce:

- `tab_th_levels.tex` — la tabla en el mismo formato del Capítulo 5 (`tabularx`, `\hline`,
  caption debajo), lista para `\input{00Tablas/06Seccion/tab_th_levels}`;
- `tabla_niveles_cap6.csv` — los mismos números en CSV, por si los quieres para otra cosa;
- un diagnóstico de las unidades excluidas por datos faltantes y una comprobación de
  **separación completa** (categorías sin ningún deslizamiento), que es lo que haría que un
  coeficiente del ZINB se fuera al infinito.

No necesita geometría: lee solo la tabla de atributos, así que corre en segundos.

## 1. Settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PATH_SU  = '/home/oisanchezp/Thesis/src/05_Chapter/slope_units_final_mm.gpkg'
OUT_DIR  = Path('00Tablas/06Seccion')
TEX_NAME = 'tab_th_levels.tex'
CSV_NAME = 'tabla_niveles_cap6.csv'

# Variables que definen el universo modelado: una unidad entra solo si las
# cuatro estan completas.
VARS = ['suelos', 'cobertura_cat', 'su_type', 'zona_lluvia']

# Categorias de referencia de los efectos FIJOS. La zona de lluvia no tiene,
# porque entra como intercepto aleatorio.
REFERENCE = {
    'Soil type':       'Sandy silt',
    'Land cover':      'Forests',
    'Slope unit type': 'Steep concave–convex slopes',
    'Rainfall zone':   None,
}

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Read the attribute table

Sin geometría: `read_geometry=False` en pyogrio, o `ignore_geometry=True` en geopandas.
Leer los 61,073 polígonos para contar filas sería tirar un minuto a la basura.

In [2]:
try:
    from pyogrio import read_dataframe
    su = read_dataframe(PATH_SU, read_geometry=False)
except Exception:
    import geopandas as gpd
    su = gpd.read_file(PATH_SU, ignore_geometry=True)

print(f'Rows    : {len(su):,}')
print(f'Columns : {list(su.columns)}')

Rows    : 61,073
Columns : ['Id', 'gridcode', 'Shape_Leng', 'Shape_Area', 'slope_mean', 'curva_mean', 'acos_mean', 'asin_mean', 'aspect_mea', 'area', 'su_type', 'su_id', 'Codigo', 'Unidad_geo', 'Granulomet', 'Espesor_m', 'Gamma_kNm3', 'C_kPa', 'Phi', 'cobertura_cat', 'cobertura', 'n_mm', 'si_no', 'suelos', 'zona_lluvia', 'curva_std_mean', 'curva_std_std', 'cluster_prob']


## 3. Labels in English

In [3]:
COVER_FROM_CODE = {
    0: 'Continuous urban fabric',
    1: 'Discontinuous urban fabric',
    2: 'Agricultural areas',
    3: 'Forests',
    4: 'Bare and degraded land',
}
COVER_FROM_ES = {
    'Tejido urbano continuo':        'Continuous urban fabric',
    'Tejido urbano discontinuo':     'Discontinuous urban fabric',
    'Territorios agricolas':         'Agricultural areas',
    'Territorios agrícolas':    'Agricultural areas',
    'Bosques':                       'Forests',
    'Tierras desnudas y degradadas': 'Bare and degraded land',
}
SOIL_FROM_ES = {
    'Limo-arcilloso':                  'Silty clay',
    'Areno-arcilloso':                 'Sandy clay',
    'Areno-limoso':                    'Sandy silt',
    'Bloques, gravas, arenas y lodos': 'Blocks, gravel, sand and mud',
}
SU_LABELS = {
    1: 'Steep concave–convex slopes',
    2: 'Intermediate concave–convex slopes',
    3: 'Steep concave slopes',
}
# Mapeo verificado contra la geografia (ver notebook de la figura).
ZONE_LABELS = {
    1: 'Oriente',
    2: 'Norte',
    3: 'Sur',
    4: 'San Antonio de Prado',
    5: 'Occidente',
}


def to_english(series, from_code=None, from_es=None):
    """Translate a categorical column to English, whether numeric or text."""
    if pd.api.types.is_numeric_dtype(series) and from_code is not None:
        return series.map(lambda v: from_code.get(int(v)) if pd.notna(v) else np.nan)
    s = series.astype(str).str.strip()
    s = s.where(series.notna(), np.nan)
    return s.map(from_es)


def to_label_from_code(series, mapping):
    """Map a numeric code column to labels."""
    if pd.api.types.is_numeric_dtype(series):
        return series.map(lambda v: mapping.get(int(v)) if pd.notna(v) else np.nan)
    s = series.astype(str).str.strip()
    out = pd.to_numeric(s, errors='coerce').map(
        lambda v: mapping.get(int(v)) if pd.notna(v) else np.nan)
    canon = {str(v).strip().lower(): v for v in mapping.values()}
    canon['sap'] = mapping.get(4, 'San Antonio de Prado')
    return out.fillna(s.str.lower().map(canon)).where(series.notna(), np.nan)


su['soil_en']  = to_english(su['suelos'], None, SOIL_FROM_ES)
su['cover_en'] = to_english(su['cobertura_cat'], COVER_FROM_CODE, COVER_FROM_ES)
su['su_en']    = to_label_from_code(su['su_type'], SU_LABELS)
su['zone_en']  = to_label_from_code(su['zona_lluvia'], ZONE_LABELS)

for src, dst in [('suelos', 'soil_en'), ('cobertura_cat', 'cover_en'),
                 ('su_type', 'su_en'), ('zona_lluvia', 'zone_en')]:
    bad = su.loc[su[dst].isna() & su[src].notna(), src].unique()
    print(f'{"WARNING - unmapped in " + src + ": " + str(bad) if len(bad) else "OK - " + src}')

OK - suelos
OK - cobertura_cat
OK - su_type
OK - zona_lluvia


## 4. Missing data: how many units are dropped, and does it matter?

Excluir unidades por datos faltantes es inocuo solo si se cumplen dos condiciones: que las
excluidas no estén concentradas en una zona concreta, y que no se lleven deslizamientos
consigo. Esta celda comprueba las dos y te imprime la frase que puedes llevar al texto.

In [4]:
n_total = len(su)
miss = {c: int(su[c].isna().sum()) for c in VARS}
complete = su[VARS].notna().all(axis=1)
n_model = int(complete.sum())
n_drop = n_total - n_model

print('Valores faltantes por variable:')
for c, n in miss.items():
    print(f'   {c:15s} {n}')

both = int((su['zona_lluvia'].isna() & su['cobertura_cat'].isna()).sum())
print(f'\nSin zona Y sin cobertura a la vez : {both}')
print(f'Unidades delineadas               : {n_total:,}')
print(f'Unidades excluidas                : {n_drop:,} ({100*n_drop/n_total:.2f} %)')
print(f'Unidades que entran al modelo     : {n_model:,}')

if 'n_mm' in su.columns:
    lost = int(su.loc[~complete, 'n_mm'].sum())
    print(f'Deslizamientos en las excluidas   : {lost}')
    if lost == 0:
        print('   -> ninguna unidad excluida contiene deslizamientos: la exclusion '
              'no le quita informacion al inventario.')
    else:
        print('   -> OJO: se pierden deslizamientos. Dilo explicitamente en el texto '
              'y considera imputar la categoria faltante en vez de eliminar.')

su = su[complete].copy()
print(f'\nTrabajando con {len(su):,} unidades.')

Valores faltantes por variable:
   suelos          0
   cobertura_cat   95
   su_type         0
   zona_lluvia     138

Sin zona Y sin cobertura a la vez : 74
Unidades delineadas               : 61,073
Unidades excluidas                : 159 (0.26 %)
Unidades que entran al modelo     : 60,914
Deslizamientos en las excluidas   : 0
   -> ninguna unidad excluida contiene deslizamientos: la exclusion no le quita informacion al inventario.

Trabajando con 60,914 unidades.


## 5. Build the table

Una fila por nivel, con el número de unidades, su porcentaje, cuántas de esas unidades tienen
al menos un deslizamiento registrado y cuántos deslizamientos suman. La penúltima columna es
la que detecta la **separación completa**.

In [5]:
BLOCKS = [
    ('Soil type',       'soil_en',
     ['Sandy silt', 'Sandy clay', 'Blocks, gravel, sand and mud', 'Silty clay']),
    ('Land cover',      'cover_en',
     ['Forests', 'Agricultural areas', 'Discontinuous urban fabric',
      'Continuous urban fabric', 'Bare and degraded land']),
    ('Slope unit type', 'su_en',    [SU_LABELS[k] for k in sorted(SU_LABELS)]),
    ('Rainfall zone',   'zone_en',
     ['Norte', 'Sur', 'Occidente', 'Oriente', 'San Antonio de Prado']),
]

rows = []
for var, col, cats in BLOCKS:
    for cat in cats:
        sub = su[su[col] == cat]
        if len(sub) == 0:
            continue
        rows.append({
            'Variable':   var,
            'Level':      cat,
            'Units':      len(sub),
            'Percent':    round(100 * len(sub) / len(su), 1),
            'WithLs':     int((sub['n_mm'] > 0).sum()) if 'n_mm' in su.columns else np.nan,
            'Landslides': int(sub['n_mm'].sum()) if 'n_mm' in su.columns else np.nan,
            'Reference':  REFERENCE.get(var) == cat,
        })

tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
tab.to_csv(OUT_DIR / CSV_NAME, index=False)

empty = tab[tab['WithLs'] == 0]
if len(empty):
    print('\nAVISO - separacion completa: estas categorias no tienen ningun '
          'deslizamiento observado, y su coeficiente en el ZINB se ira al infinito:')
    print(empty[['Variable', 'Level', 'Units']].to_string(index=False))
    print('Agrupalas con la categoria mas parecida o reportalas explicitamente.')
else:
    print('\nOK - todas las categorias tienen al menos un deslizamiento observado: '
          'no hay separacion completa.')

       Variable                              Level  Units  Percent  WithLs  Landslides  Reference
      Soil type                         Sandy silt  23196     38.1    1411        2001       True
      Soil type                         Sandy clay  17125     28.1    1316        2151      False
      Soil type       Blocks, gravel, sand and mud  11731     19.3     606         936      False
      Soil type                         Silty clay   8862     14.5     727        1110      False
     Land cover                            Forests  27148     44.6    1941        2958       True
     Land cover                 Agricultural areas  17740     29.1    1123        1738      False
     Land cover         Discontinuous urban fabric  11845     19.4     712        1113      False
     Land cover            Continuous urban fabric   3744      6.1     253         342      False
     Land cover             Bare and degraded land    437      0.7      31          47      False
Slope unit type     

## 6. Write the LaTeX table

Mismo formato que las tablas del Capítulo 5: `tabularx`, `\hline`, `\footnotesize` y caption
debajo. El nombre de la variable aparece solo en la primera fila de cada bloque, y la
categoría de referencia va marcada con un asterisco.

In [6]:
def tex_escape(s):
    """En dashes y caracteres que LaTeX interpreta."""
    s = s.replace('–', '--').replace('—', '---')
    for a, b in [('&', r'\&'), ('%', r'\%'), ('_', r'\_'), ('#', r'\#')]:
        s = s.replace(a, b)
    return s


def num(v):
    """Miles con {,} y en modo matematico, como en las tablas del Cap. 4."""
    return '$' + f'{int(v):,}'.replace(',', '{,}') + '$'


lines = []
lines.append(r'\renewcommand{\arraystretch}{1.2}')
lines.append(r'\setlength{\tabcolsep}{5pt}')
lines.append('')
lines.append(r'\begin{table}[ht]')
lines.append(r'\centering')
lines.append(r'\footnotesize')
lines.append(r'\begin{tabularx}{\textwidth}'
             r'{>{\raggedright\arraybackslash}m{2.4cm} X r r r r}')
lines.append(r'\hline')
lines.append(r'\textbf{Variable} & \textbf{Level} & \textbf{Slope units} & '
             r'\textbf{\%} & \textbf{With $Y_i>0$} & \textbf{Landslides} \\')
lines.append(r'\hline')

prev = None
for _, r in tab.iterrows():
    head = tex_escape(r['Variable']) if r['Variable'] != prev else ''
    prev = r['Variable']
    lvl = tex_escape(r['Level']) + (r'$^{*}$' if r['Reference'] else '')
    lines.append(f"{head} & {lvl} & {num(r['Units'])} & {r['Percent']:.1f} & "
                 f"{num(r['WithLs'])} & {num(r['Landslides'])} " + r'\\')

lines.append(r'\hline')
lines.append(r'\end{tabularx}')

n_fmt = '$' + f'{len(su):,}'.replace(',', '{,}') + '$'
lines.append(
    r'\caption{Levels of the four categorical predisposing variables, number of slope '
    r'units per level, number of slope units with at least one recorded landslide and '
    r'total number of landslides. Percentages are computed over the ' + n_fmt +
    r' slope units that entered the models. Reference categories of the fixed effects '
    r'are marked with an asterisk; the rainfall zone has no reference category because '
    r'it enters the model as a random intercept.}')
lines.append(r'\label{tab:th-levels}')
lines.append(r'\end{table}')

tex = '\n'.join(lines) + '\n'
(OUT_DIR / TEX_NAME).write_text(tex, encoding='utf-8')

print(tex)
print('Guardado en:', OUT_DIR / TEX_NAME)

\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{5pt}

\begin{table}[ht]
\centering
\footnotesize
\begin{tabularx}{\textwidth}{>{\raggedright\arraybackslash}m{2.4cm} X r r r r}
\hline
\textbf{Variable} & \textbf{Level} & \textbf{Slope units} & \textbf{\%} & \textbf{With $Y_i>0$} & \textbf{Landslides} \\
\hline
Soil type & Sandy silt$^{*}$ & $23{,}196$ & 38.1 & $1{,}411$ & $2{,}001$ \\
 & Sandy clay & $17{,}125$ & 28.1 & $1{,}316$ & $2{,}151$ \\
 & Blocks, gravel, sand and mud & $11{,}731$ & 19.3 & $606$ & $936$ \\
 & Silty clay & $8{,}862$ & 14.5 & $727$ & $1{,}110$ \\
Land cover & Forests$^{*}$ & $27{,}148$ & 44.6 & $1{,}941$ & $2{,}958$ \\
 & Agricultural areas & $17{,}740$ & 29.1 & $1{,}123$ & $1{,}738$ \\
 & Discontinuous urban fabric & $11{,}845$ & 19.4 & $712$ & $1{,}113$ \\
 & Continuous urban fabric & $3{,}744$ & 6.1 & $253$ & $342$ \\
 & Bare and degraded land & $437$ & 0.7 & $31$ & $47$ \\
Slope unit type & Steep concave--convex slopes$^{*}$ & $31{,}561$ & 51.8 & $2{,

## 7. How to use it in Overleaf

En el preámbulo tienes que tener `tabularx` (ya lo usas en el Capítulo 5):

```latex
\usepackage{tabularx}
```

Y en el capítulo, donde va la tabla:

```latex
\input{00Tablas/06Seccion/tab_th_levels}
```

Si prefieres pegar la tabla en vez de enlazarla, cópiala de la salida de la celda anterior.
Pero enlazarla es mejor: cada vez que corras el notebook, la tabla del PDF se actualiza sola.